# Step 2 — Curated Operational Dataset and KPI Foundations

This notebook builds the star schema analytical model and derives KPI flags for the Power BI dashboard.

**Outputs:** `fact_flights.parquet`, `dim_airlines.csv`, `dim_airports.csv`, `dim_cancellation_codes.csv`

In [1]:
from __future__ import annotations

import os
from pathlib import Path
import pandas as pd
import numpy as np
import re

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 80)

print("Imports loaded.")

Imports loaded.


## 01 — Configuration and Paths

In [2]:
# Project root (Windows)
PROJECT_ROOT = Path(r"C:\Users\kados\OneDrive\Desktop\folders\Data Analysis Portfolio\Airline Flight Delays Portfolio")

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"

RAW_FLIGHTS = RAW_DIR / "flights.csv"
RAW_AIRLINES = RAW_DIR / "airlines.csv"
RAW_AIRPORTS = RAW_DIR / "airports.csv"
RAW_CANCEL_CODES = RAW_DIR / "cancellation_codes.csv"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"RAW_FLIGHTS exists: {RAW_FLIGHTS.exists()}")
print(f"PROCESSED_DIR: {PROCESSED_DIR}")

PROJECT_ROOT: C:\Users\kados\OneDrive\Desktop\folders\Data Analysis Portfolio\Airline Flight Delays Portfolio
RAW_FLIGHTS exists: True
PROCESSED_DIR: C:\Users\kados\OneDrive\Desktop\folders\Data Analysis Portfolio\Airline Flight Delays Portfolio\data\processed


In [3]:
RUN_ID = pd.Timestamp.now(tz="UTC").strftime("%Y%m%d_%H%M%S")
print(f"RUN_ID: {RUN_ID}")

RUN_ID: 20260621_213627


## 02 — Utility Functions

In [4]:
def save_csv(df: pd.DataFrame, path: Path, *, index: bool = False) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=index)
    print(f"Saved CSV: {path} | rows={len(df):,} cols={df.shape[1]:,}")

def save_parquet(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=False)
    print(f"Saved Parquet: {path} | rows={len(df):,} cols={df.shape[1]:,}")

## 03 — Load Lookup Dimensions

In [5]:
airlines = pd.read_csv(RAW_AIRLINES)
airports = pd.read_csv(RAW_AIRPORTS)
cancel_codes = pd.read_csv(RAW_CANCEL_CODES)

print(f"airlines: {airlines.shape}")
print(f"airports: {airports.shape}")
print(f"cancel_codes: {cancel_codes.shape}")

display(airlines.head(3))
display(airports.head(3))
display(cancel_codes.head(3))

airlines: (14, 2)
airports: (322, 7)
cancel_codes: (4, 2)


,IATA_CODE,AIRLINE
0,UA,United Air Lines Inc.
1,AA,American Airlines Inc.
2,US,US Airways Inc.


,IATA_CODE,AIRPORT,CITY,STATE,COUNTRY,LATITUDE,LONGITUDE
0,ABE,Lehigh Valley International Airport,Allentown,PA,USA,40.65236,-75.44040
1,ABI,Abilene Regional Airport,Abilene,TX,USA,32.41132,-99.68190
2,ABQ,Albuquerque International Sunport,Albuquerque,NM,USA,35.04022,-106.60919


,CANCELLATION_REASON,CANCELLATION_DESCRIPTION
0,A,Airline/Carrier
1,B,Weather
2,C,National Air System


## 04 — Load Flights with Controlled Schema

In [6]:
FACT_USECOLS = [
    "YEAR", "MONTH", "DAY",
    "AIRLINE", "FLIGHT_NUMBER",
    "ORIGIN_AIRPORT", "DESTINATION_AIRPORT",
    "SCHEDULED_DEPARTURE",

    # Optional drilldown
    "TAIL_NUMBER",

    # Status
    "CANCELLED", "CANCELLATION_REASON", "DIVERTED",

    # Delays
    "DEPARTURE_DELAY", "ARRIVAL_DELAY",

    # Delay components
    "AIR_SYSTEM_DELAY", "SECURITY_DELAY", "AIRLINE_DELAY", "LATE_AIRCRAFT_DELAY", "WEATHER_DELAY",

    # Ops measures
    "DISTANCE", "SCHEDULED_TIME", "ELAPSED_TIME", "AIR_TIME", "TAXI_OUT", "TAXI_IN",
]

In [7]:
DTYPE_MAP = {
    # Identity
    "YEAR": "Int16",
    "MONTH": "Int8",
    "DAY": "Int8",
    "FLIGHT_NUMBER": "Int32",

    # Flags
    "CANCELLED": "Int8",
    "DIVERTED": "Int8",

    # Codes
    "AIRLINE": "category",
    "ORIGIN_AIRPORT": "category",
    "DESTINATION_AIRPORT": "category",
    "CANCELLATION_REASON": "category",
    "TAIL_NUMBER": "category",

    # Delays
    "DEPARTURE_DELAY": "Int32",
    "ARRIVAL_DELAY": "Int32",

    # Delay components
    "AIR_SYSTEM_DELAY": "Int32",
    "SECURITY_DELAY": "Int32",
    "AIRLINE_DELAY": "Int32",
    "LATE_AIRCRAFT_DELAY": "Int32",
    "WEATHER_DELAY": "Int32",

    # Times
    "SCHEDULED_DEPARTURE": "Int32",
    "DEPARTURE_TIME": "Int32",
    "SCHEDULED_ARRIVAL": "Int32",
    "ARRIVAL_TIME": "Int32",
    "TAXI_OUT": "Int16",
    "SCHEDULED_TIME": "Int16",
    "ELAPSED_TIME": "Int16",
    "AIR_TIME": "Int16",
    "DISTANCE": "Int16",
    "WHEELS_ON": "Int32",
    "WHEELS_OFF": "Int32",
    "TAXI_IN": "Int16",
}

In [8]:
df = pd.read_csv(
    RAW_FLIGHTS,
    usecols=FACT_USECOLS,
    dtype=DTYPE_MAP
)

print(f"Loaded df: {df.shape}")
print(f"Memory (MB): {round(df.memory_usage(deep=True).sum() / 1024**2, 2)}")

Loaded df: (5819079, 25)
Memory (MB): 455.3


## 05 — Derive Time Slicing Columns

In [9]:
# Create flight_date
df["flight_date"] = pd.to_datetime(
    dict(year=df["YEAR"], month=df["MONTH"], day=df["DAY"]),
    errors="coerce"
).dt.date

# Derive scheduled departure hour from HHMM integer
sched = df["SCHEDULED_DEPARTURE"].astype("int32")
df["scheduled_departure_hour"] = (sched // 100).astype("int8")

# Day and month labels
dt = pd.to_datetime(df["flight_date"])
df["day_of_week"] = dt.dt.day_name()
df["month"] = dt.dt.month_name()

# Time band buckets
h = df["scheduled_departure_hour"]
df["time_band"] = pd.cut(
    h,
    bins=[-1, 5, 9, 15, 19, 23],
    labels=["EarlyAM", "AMPeak", "Midday", "PMPeak", "Evening"]
).astype("category")

df[[
    "flight_date",
    "SCHEDULED_DEPARTURE",
    "scheduled_departure_hour",
    "time_band",
    "day_of_week",
    "month"
]].head()

,flight_date,SCHEDULED_DEPARTURE,scheduled_departure_hour,time_band,day_of_week,month
0,2015-01-01,5,0,EarlyAM,Thursday,January
1,2015-01-01,10,0,EarlyAM,Thursday,January
2,2015-01-01,20,0,EarlyAM,Thursday,January
3,2015-01-01,20,0,EarlyAM,Thursday,January
4,2015-01-01,25,0,EarlyAM,Thursday,January


## 06 — Derive KPI Flags

In [10]:
# Eligibility: completed flights only (not cancelled, not diverted)
df["is_otp15_eligible"] = (
    (df["CANCELLED"] == 0) &
    (df["DIVERTED"] == 0)
).astype("Int8")

# On-Time Performance (OTP15): <= 15 mins arrival delay, only for eligible flights
df["is_on_time_otp15"] = (
    (df["is_otp15_eligible"] == 1) &
    (df["ARRIVAL_DELAY"] <= 15)
).astype("Int8")

# Delayed (OTP15): > 15 mins arrival delay, only for eligible flights
df["is_delayed_otp15"] = (
    (df["is_otp15_eligible"] == 1) &
    (df["ARRIVAL_DELAY"] > 15)
).astype("Int8")

# Severe Delay (SD60): >= 60 mins arrival delay, only for eligible flights
df["is_severe_delay_sd60"] = (
    (df["is_otp15_eligible"] == 1) &
    (df["ARRIVAL_DELAY"] >= 60)
).astype("Int8")

In [11]:
df[[
    "is_otp15_eligible",
    "is_on_time_otp15",
    "is_delayed_otp15",
    "is_severe_delay_sd60"
]].mean()

is_otp15_eligible       0.981944
is_on_time_otp15        0.806057
is_delayed_otp15        0.175887
is_severe_delay_sd60    0.055952
dtype: Float64

In [12]:
eligible = df["is_otp15_eligible"] == 1

otp15 = df.loc[eligible, "is_on_time_otp15"].mean()
severe_rate = df.loc[eligible, "is_severe_delay_sd60"].mean()

print(f"OTP15 % (eligible only): {round(otp15 * 100, 2)}%")
print(f"Severe Delay % (eligible only): {round(severe_rate * 100, 2)}%")

OTP15 % (eligible only): 82.09%
Severe Delay % (eligible only): 5.7%


## 07 — Derive Delay Buckets and Extreme Flag

In [13]:
# Delay bucket (only meaningful for eligible flights, but we classify all)

df["delay_bucket"] = pd.cut(
    df["ARRIVAL_DELAY"],
    bins=[-10_000, 15, 30, 59, 10_000],
    labels=["<=15 (On-time)", "16-30", "31-59", "60+"]
).astype("category")

# Extreme arrival delay flag (based on Step 1 discovery)
df["is_extreme_arrival_delay"] = (
    (df["ARRIVAL_DELAY"] < -60) |
    (df["ARRIVAL_DELAY"] > 1440)
).astype("Int8")

In [14]:
print("Delay bucket distribution:")
print(df["delay_bucket"].value_counts(normalize=True).round(4))
print(f"Extreme arrival delay rate: {df['is_extreme_arrival_delay'].mean():.6f}")

Delay bucket distribution:
delay_bucket
<=15 (On-time)    0.8209
16-30             0.0684
60+               0.0570
31-59             0.0537
Name: proportion, dtype: float64
Extreme arrival delay rate: 0.000074


## 08 — Derive Primary Delay Driver

In [15]:
delay_components = [
    "AIR_SYSTEM_DELAY",
    "SECURITY_DELAY",
    "AIRLINE_DELAY",
    "LATE_AIRCRAFT_DELAY",
    "WEATHER_DELAY"
]

comp_df = df[delay_components]

# Sum across components (all-NA -> 0, all-zero -> 0)
component_sum = comp_df.fillna(0).sum(axis=1)

# Max minutes 
df["primary_delay_driver_minutes"] = comp_df.fillna(0).max(axis=1)

# Safe idxmax: no error because we removed NA for comparison
df["primary_delay_driver"] = comp_df.fillna(0).idxmax(axis=1)

# Not delayed => not applicable (NULL)
not_delayed = df["is_delayed_otp15"] == 0
df.loc[not_delayed, "primary_delay_driver"] = pd.NA
df.loc[not_delayed, "primary_delay_driver_minutes"] = pd.NA

# Delayed but no attribution (all components 0 or NA) => Unattributed
delayed = df["is_delayed_otp15"] == 1
no_attribution = delayed & (component_sum == 0)

df.loc[no_attribution, "primary_delay_driver"] = "Unattributed"
df.loc[no_attribution, "primary_delay_driver_minutes"] = pd.NA

# Memory friendly
df["primary_delay_driver"] = df["primary_delay_driver"].astype("category")
print("Primary Delay Driver distribution (all):")
print(df["primary_delay_driver"].value_counts(dropna=False).head(10))
print("\nPrimary Delay Driver distribution (delayed only):")
print(df.loc[df["is_delayed_otp15"] == 1, "primary_delay_driver"].value_counts(normalize=True).round(3))

Primary Delay Driver distribution (all):
primary_delay_driver
NaN                    4795581
LATE_AIRCRAFT_DELAY     400438
AIRLINE_DELAY           299858
AIR_SYSTEM_DELAY        286330
WEATHER_DELAY            35059
SECURITY_DELAY            1813
Name: count, dtype: int64

Primary Delay Driver distribution (delayed only):
primary_delay_driver
LATE_AIRCRAFT_DELAY    0.391
AIRLINE_DELAY          0.293
AIR_SYSTEM_DELAY       0.280
WEATHER_DELAY          0.034
SECURITY_DELAY         0.002
Name: proportion, dtype: float64


In [16]:
dim_delay_driver = pd.DataFrame({
    "delay_driver_code": [
        "LATE_AIRCRAFT_DELAY",
        "AIRLINE_DELAY",
        "AIR_SYSTEM_DELAY",
        "WEATHER_DELAY",
        "SECURITY_DELAY",
        "Unattributed"
    ],
    "delay_driver_display": [
        "Late Aircraft",
        "Airline",
        "Air System",
        "Weather",
        "Security",
        "Unattributed"
    ],
    "sort_order": [1, 2, 3, 4, 5, 6]
})

display(dim_delay_driver)

,delay_driver_code,delay_driver_display,sort_order
0,LATE_AIRCRAFT_DELAY,Late Aircraft,1
1,AIRLINE_DELAY,Airline,2
2,AIR_SYSTEM_DELAY,Air System,3
3,WEATHER_DELAY,Weather,4
4,SECURITY_DELAY,Security,5
5,Unattributed,Unattributed,6


## 09 — Derive Operational Disruption Flag

In [17]:
df["operational_disruption_flag"] = (
    (df["CANCELLED"] == 1) |
    (df["DIVERTED"] == 1) |
    ((df["is_otp15_eligible"] == 1) & (df["is_severe_delay_sd60"] == 1))
).astype("Int8")
print(f"Operational Disruption %: {round(df['operational_disruption_flag'].mean() * 100, 2)}%")

Operational Disruption %: 7.4%


## 10 — Build Fact Table

In [18]:
fact_cols = [
    # Keys / identity
    "flight_date",
    "AIRLINE", "FLIGHT_NUMBER",
    "ORIGIN_AIRPORT", "DESTINATION_AIRPORT",
    "SCHEDULED_DEPARTURE",
    "TAIL_NUMBER",

    # Time slicers
    "scheduled_departure_hour", "time_band", "day_of_week", "month",

    # Status
    "CANCELLED", "CANCELLATION_REASON", "DIVERTED",

    # Measures
    "DISTANCE", "SCHEDULED_TIME", "ELAPSED_TIME", "AIR_TIME", "TAXI_OUT", "TAXI_IN",
    "DEPARTURE_DELAY", "ARRIVAL_DELAY",

    # Delay components
    "AIR_SYSTEM_DELAY", "SECURITY_DELAY", "AIRLINE_DELAY", "LATE_AIRCRAFT_DELAY", "WEATHER_DELAY",

    # KPI flags / features
    "is_otp15_eligible",
    "is_on_time_otp15",
    "is_delayed_otp15",
    "is_severe_delay_sd60",
    "delay_bucket",
    "is_extreme_arrival_delay",
    "primary_delay_driver",
    "primary_delay_driver_minutes",
    "operational_disruption_flag",
]

fact_flights = df[fact_cols].copy()

print(f"fact_flights shape: {fact_flights.shape}")
print(f"Memory (MB): {round(fact_flights.memory_usage(deep=True).sum() / 1024**2, 2)}")
fact_flights.head(3)

fact_flights shape: (5819079, 36)
Memory (MB): 917.16


,flight_date,AIRLINE,FLIGHT_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,TAIL_NUMBER,scheduled_departure_hour,time_band,day_of_week,month,CANCELLED,CANCELLATION_REASON,DIVERTED,DISTANCE,SCHEDULED_TIME,ELAPSED_TIME,AIR_TIME,TAXI_OUT,TAXI_IN,DEPARTURE_DELAY,ARRIVAL_DELAY,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY,is_otp15_eligible,is_on_time_otp15,is_delayed_otp15,is_severe_delay_sd60,delay_bucket,is_extreme_arrival_delay,primary_delay_driver,primary_delay_driver_minutes,operational_disruption_flag
0,2015-01-01,AS,98,ANC,SEA,5,N407AS,0,EarlyAM,Thursday,January,0,NaN,0,1448,205,194,169,21,4,-11,-22,<NA>,<NA>,<NA>,<NA>,<NA>,1,1,0,0,<=15 (On-time),0,NaN,<NA>,0
1,2015-01-01,AA,2336,LAX,PBI,10,N3KUAA,0,EarlyAM,Thursday,January,0,NaN,0,2330,280,279,263,12,4,-8,-9,<NA>,<NA>,<NA>,<NA>,<NA>,1,1,0,0,<=15 (On-time),0,NaN,<NA>,0
2,2015-01-01,US,840,SFO,CLT,20,N171US,0,EarlyAM,Thursday,January,0,NaN,0,2296,286,293,266,16,11,-2,5,<NA>,<NA>,<NA>,<NA>,<NA>,1,1,0,0,<=15 (On-time),0,NaN,<NA>,0


## 11 — Prepare Dimension Tables

In [19]:
dim_airlines = pd.read_csv(RAW_AIRLINES)
dim_airlines.columns = [c.strip().lower() for c in dim_airlines.columns]
dim_airlines = dim_airlines.rename(columns={
    "iata_code": "airline_code",
    "airline": "airline_name"
})

dim_airports = pd.read_csv(RAW_AIRPORTS)
dim_airports.columns = [c.strip().lower() for c in dim_airports.columns]
dim_airports = dim_airports.rename(columns={
    "iata_code": "airport_code",
    "airport": "airport_name"
})

dim_cancel = pd.read_csv(RAW_CANCEL_CODES)
dim_cancel.columns = [c.strip().lower() for c in dim_cancel.columns]
dim_cancel = dim_cancel.rename(columns={
    "cancellation_reason": "cancellation_code",
    "description": "cancellation_description"
})

print(f"dim_airlines: {dim_airlines.shape} | cols: {dim_airlines.columns.tolist()}")
print(f"dim_airports: {dim_airports.shape} | cols: {dim_airports.columns.tolist()}")
print(f"dim_cancel: {dim_cancel.shape} | cols: {dim_cancel.columns.tolist()}")

display(dim_airlines.head(3))
display(dim_airports.head(3))
display(dim_cancel.head(3))

dim_airlines: (14, 2) | cols: ['airline_code', 'airline_name']
dim_airports: (322, 7) | cols: ['airport_code', 'airport_name', 'city', 'state', 'country', 'latitude', 'longitude']
dim_cancel: (4, 2) | cols: ['cancellation_code', 'cancellation_description']


,airline_code,airline_name
0,UA,United Air Lines Inc.
1,AA,American Airlines Inc.
2,US,US Airways Inc.


,airport_code,airport_name,city,state,country,latitude,longitude
0,ABE,Lehigh Valley International Airport,Allentown,PA,USA,40.65236,-75.44040
1,ABI,Abilene Regional Airport,Abilene,TX,USA,32.41132,-99.68190
2,ABQ,Albuquerque International Sunport,Albuquerque,NM,USA,35.04022,-106.60919


,cancellation_code,cancellation_description
0,A,Airline/Carrier
1,B,Weather
2,C,National Air System


## 12 — Referential Integrity Check

In [20]:
# 1) Airline codes in fact must exist in dim_airlines
missing_airlines = (
    set(fact_flights["AIRLINE"].dropna().unique())
    - set(dim_airlines["airline_code"].dropna().unique())
)

# 2) Airport codes in fact must exist in dim_airports (origin + dest)
missing_origin_airports = (
    set(fact_flights["ORIGIN_AIRPORT"].dropna().unique())
    - set(dim_airports["airport_code"].dropna().unique())
)
missing_dest_airports = (
    set(fact_flights["DESTINATION_AIRPORT"].dropna().unique())
    - set(dim_airports["airport_code"].dropna().unique())
)

# 3) Cancellation codes in fact must exist in dim_cancel (only where cancelled=1)
cancelled_codes = fact_flights.loc[fact_flights["CANCELLED"] == 1, "CANCELLATION_REASON"]
missing_cancel_codes = (
    set(cancelled_codes.dropna().unique())
    - set(dim_cancel["cancellation_code"].dropna().unique())
)

print(f"Missing airline codes: {len(missing_airlines)} {sorted(list(missing_airlines))[:10]}")
print(f"Missing origin airport codes: {len(missing_origin_airports)} {sorted(list(missing_origin_airports))[:10]}")
print(f"Missing dest airport codes: {len(missing_dest_airports)} {sorted(list(missing_dest_airports))[:10]}")
print(f"Missing cancellation codes: {len(missing_cancel_codes)} {sorted(list(missing_cancel_codes))[:10]}")

Missing airline codes: 0 []
Missing origin airport codes: 306 ['10135', '10136', '10140', '10141', '10146', '10154', '10155', '10157', '10158', '10165']
Missing dest airport codes: 307 ['10135', '10136', '10140', '10141', '10146', '10154', '10155', '10157', '10158', '10165']
Missing cancellation codes: 0 []


## 13 — Fix Missing Airport Codes

In [21]:
fact_flights["ORIGIN_AIRPORT"] = fact_flights["ORIGIN_AIRPORT"].astype(str)
fact_flights["DESTINATION_AIRPORT"] = fact_flights["DESTINATION_AIRPORT"].astype(str)
dim_airports["airport_code"] = dim_airports["airport_code"].astype(str)

In [22]:
missing_codes = sorted(list(
    (set(fact_flights["ORIGIN_AIRPORT"].unique()) | set(fact_flights["DESTINATION_AIRPORT"].unique()))
    - set(dim_airports["airport_code"].unique())
))

missing_airports_rows = pd.DataFrame({
    "airport_code": missing_codes,
    "airport_name": pd.NA,
    "city": pd.NA,
    "state": pd.NA,
    "country": pd.NA,
    "latitude": pd.NA,
    "longitude": pd.NA,
})

dim_airports_fixed = pd.concat([dim_airports, missing_airports_rows], ignore_index=True)
print(f"dim_airports_fixed: {dim_airports_fixed.shape}")
print(f"Added missing airport codes: {len(missing_codes)}")

dim_airports_fixed: (629, 7)
Added missing airport codes: 307


In [23]:
print("Airport DOT → IATA mapping")

airport_lookup_DOT = pd.read_csv(
    PROJECT_ROOT / "data/raw/L_AIRPORT_ID.csv",
    names=["DOT_ID", "AIRPORT_INFO"],
    skiprows=1
)

display(airport_lookup_DOT.head())

Airport DOT → IATA mapping


,DOT_ID,AIRPORT_INFO
0,10001,"Afognak Lake, AK: Afognak Lake Airport"
1,10003,"Granite Mountain, AK: Bear Creek Mining Strip"
2,10004,"Lik, AK: Lik Mining Camp"
3,10005,"Little Squaw, AK: Little Squaw Airport"
4,10006,"Kizhuyak, AK: Kizhuyak Bay"


In [24]:
print(airport_lookup_DOT.dtypes)

DOT_ID          int64
AIRPORT_INFO      str
dtype: object


In [25]:
print("Null DOT_ID:", airport_lookup_DOT["DOT_ID"].isna().sum())
print("Null AIRPORT_INFO:", airport_lookup_DOT["AIRPORT_INFO"].isna().sum())
print("Duplicate DOT_ID:", airport_lookup_DOT["DOT_ID"].duplicated().sum())
print("Row count:", len(airport_lookup_DOT))

Null DOT_ID: 0
Null AIRPORT_INFO: 0
Duplicate DOT_ID: 0
Row count: 6850


In [26]:
numeric_origin = fact_flights["ORIGIN_AIRPORT"].astype(str).str.isnumeric()
numeric_dest = fact_flights["DESTINATION_AIRPORT"].astype(str).str.isnumeric()

print("Numeric origin rows:", int(numeric_origin.sum()))
print("Numeric destination rows:", int(numeric_dest.sum()))

print("Unique numeric origin IDs:", fact_flights.loc[numeric_origin, "ORIGIN_AIRPORT"].nunique())
print("Unique numeric destination IDs:", fact_flights.loc[numeric_dest, "DESTINATION_AIRPORT"].nunique())

Numeric origin rows: 486165
Numeric destination rows: 486165
Unique numeric origin IDs: 306
Unique numeric destination IDs: 307


## Airport code repair

Some airport values in the fact table are numeric DOT airport IDs, while the airport dimension uses IATA-style airport codes. This caused unmatched relationships in Power BI and produced blank airport names.

This step:
- preserves the original raw airport values,
- maps numeric DOT IDs to known airport codes where the match is safe,
- keeps unmatched DOT IDs as valid keys with readable airport names,
- adds mapping status fields for auditability,
- validates that every airport key in the fact table exists in the airport dimension.

In [27]:
import re
import unicodedata
from difflib import SequenceMatcher

print("Airport code repair: DOT IDs, aliases, and final dimension coverage")

# ------------------------------------------------------------
# 1. Preserve raw airport codes before cleaning/mapping
# ------------------------------------------------------------

if "ORIGIN_AIRPORT_RAW" not in fact_flights.columns:
    fact_flights["ORIGIN_AIRPORT_RAW"] = fact_flights["ORIGIN_AIRPORT"]

if "DESTINATION_AIRPORT_RAW" not in fact_flights.columns:
    fact_flights["DESTINATION_AIRPORT_RAW"] = fact_flights["DESTINATION_AIRPORT"]

fact_flights["ORIGIN_AIRPORT_RAW"] = fact_flights["ORIGIN_AIRPORT_RAW"].astype("string").str.strip()
fact_flights["DESTINATION_AIRPORT_RAW"] = fact_flights["DESTINATION_AIRPORT_RAW"].astype("string").str.strip()

dim_airports = dim_airports.copy()
dim_airports["airport_code"] = dim_airports["airport_code"].astype("string").str.strip()
dim_airports["airport_name"] = dim_airports["airport_name"].astype("string").str.strip()
dim_airports["city"] = dim_airports["city"].astype("string").str.strip()
dim_airports["state"] = dim_airports["state"].astype("string").str.strip()

airport_lookup_DOT = airport_lookup_DOT.copy()
airport_lookup_DOT["DOT_ID"] = airport_lookup_DOT["DOT_ID"].astype("string").str.strip()
airport_lookup_DOT["AIRPORT_INFO"] = airport_lookup_DOT["AIRPORT_INFO"].astype("string").str.strip()


# ------------------------------------------------------------
# 2. Normalisation helpers for name/alias matching
# ------------------------------------------------------------

def normalise_text(value, loose=False):
    if pd.isna(value):
        return pd.NA

    text = str(value)

    # ASCII-safe normalisation
    text = unicodedata.normalize("NFKD", text)
    text = text.encode("ascii", "ignore").decode("ascii")

    text = text.lower().strip()

    # Common airport-name variations
    text = text.replace("&", " and ")

    abbreviation_map = {
        r"\bint['’]?l\b": "international",
        r"\bintl\b": "international",
        r"\bnat\b": "national",
        r"\bgrtr\b": "greater",
        r"\bft\b": "fort",
        r"\bst\b": "saint",
        r"\bgen\b": "general",
        r"\bbrig\b": "brigadier",
        r"\bcapt\b": "captain",
        r"\bafb\b": "air force base",
        r"\baaf\b": "army airfield",
        r"\bmcas\b": "marine corps air station",
    }

    for pattern, replacement in abbreviation_map.items():
        text = re.sub(pattern, replacement, text)

    # Remove parenthetical additions like "(Marine Air Terminal)" in loose mode
    if loose:
        text = re.sub(r"\([^)]*\)", " ", text)

    # Remove punctuation
    text = re.sub(r"[^a-z0-9]+", " ", text)

    if loose:
        removable_words = {
            "airport", "international", "regional", "municipal",
            "field", "airfield", "terminal", "county", "metropolitan",
            "memorial", "city", "the", "at", "and",
            "air", "force", "base", "army", "marine", "corps", "station",
            "national"
        }
        tokens = [t for t in text.split() if t not in removable_words]
        text = " ".join(tokens)

    text = re.sub(r"\s+", " ", text).strip()
    return text if text else pd.NA


def build_unique_lookup(df, key_cols, value_col):
    """
    Build lookup only where the key maps to exactly one value.
    This avoids unsafe many-to-one alias matches.
    """
    temp = df.dropna(subset=key_cols + [value_col]).copy()

    grouped = (
        temp.groupby(key_cols, dropna=False)[value_col]
        .nunique()
        .reset_index(name="unique_count")
    )

    safe_keys = grouped[grouped["unique_count"] == 1][key_cols]

    safe_lookup = temp.merge(safe_keys, on=key_cols, how="inner")
    safe_lookup = safe_lookup.drop_duplicates(subset=key_cols)

    return {
        tuple(row[col] for col in key_cols): row[value_col]
        for _, row in safe_lookup.iterrows()
    }


# ------------------------------------------------------------
# 3. Parse DOT lookup into city/state/airport name
#    Example: "Chicago, IL: Chicago O'Hare International"
# ------------------------------------------------------------

dot_parts = airport_lookup_DOT["AIRPORT_INFO"].str.split(":", n=1, expand=True)

airport_lookup_DOT["dot_location"] = dot_parts[0].str.strip()
airport_lookup_DOT["dot_airport_name"] = dot_parts[1].str.strip() if dot_parts.shape[1] > 1 else pd.NA

loc_parts = airport_lookup_DOT["dot_location"].str.rsplit(",", n=1, expand=True)

airport_lookup_DOT["dot_city"] = loc_parts[0].str.strip()
airport_lookup_DOT["dot_state"] = loc_parts[1].str.strip() if loc_parts.shape[1] > 1 else pd.NA

airport_lookup_DOT["dot_name_norm"] = airport_lookup_DOT["dot_airport_name"].map(
    lambda x: normalise_text(x, loose=False)
)
airport_lookup_DOT["dot_name_loose"] = airport_lookup_DOT["dot_airport_name"].map(
    lambda x: normalise_text(x, loose=True)
)
airport_lookup_DOT["dot_city_norm"] = airport_lookup_DOT["dot_city"].map(
    lambda x: normalise_text(x, loose=True)
)
airport_lookup_DOT["dot_state_norm"] = airport_lookup_DOT["dot_state"].astype("string").str.upper().str.strip()


# ------------------------------------------------------------
# 4. Prepare airport dimension for matching
# ------------------------------------------------------------

dim_match = dim_airports.copy()

dim_match["dim_name_norm"] = dim_match["airport_name"].map(lambda x: normalise_text(x, loose=False))
dim_match["dim_name_loose"] = dim_match["airport_name"].map(lambda x: normalise_text(x, loose=True))
dim_match["dim_city_norm"] = dim_match["city"].map(lambda x: normalise_text(x, loose=True))
dim_match["dim_state_norm"] = dim_match["state"].astype("string").str.upper().str.strip()

known_airport_codes = set(dim_match["airport_code"].dropna())


# ------------------------------------------------------------
# 5. Find numeric DOT IDs that appear in fact_flights
# ------------------------------------------------------------

fact_airport_codes_raw = pd.concat([
    fact_flights["ORIGIN_AIRPORT_RAW"],
    fact_flights["DESTINATION_AIRPORT_RAW"]
]).dropna().astype("string").str.strip()

numeric_fact_codes = sorted(
    code for code in fact_airport_codes_raw.unique()
    if bool(re.fullmatch(r"\d+", str(code)))
)

numeric_dot_ids_in_lookup = set(airport_lookup_DOT["DOT_ID"])
numeric_fact_codes_in_dot_lookup = sorted(set(numeric_fact_codes) & numeric_dot_ids_in_lookup)
numeric_fact_codes_not_in_dot_lookup = sorted(set(numeric_fact_codes) - numeric_dot_ids_in_lookup)

print("Unique numeric airport codes in fact:", len(numeric_fact_codes))
print("Numeric codes found in DOT lookup:", len(numeric_fact_codes_in_dot_lookup))
print("Numeric codes NOT found in DOT lookup:", len(numeric_fact_codes_not_in_dot_lookup))


# ------------------------------------------------------------
# 6. Build DOT_ID -> IATA mapping using exact + alias + safe fuzzy matching
# ------------------------------------------------------------

dot_candidates = airport_lookup_DOT[
    airport_lookup_DOT["DOT_ID"].isin(numeric_fact_codes_in_dot_lookup)
].copy()

# Safe exact/alias lookup maps
lookup_state_city_name = build_unique_lookup(
    dim_match,
    ["dim_state_norm", "dim_city_norm", "dim_name_norm"],
    "airport_code"
)

lookup_state_name = build_unique_lookup(
    dim_match,
    ["dim_state_norm", "dim_name_norm"],
    "airport_code"
)

lookup_state_name_loose = build_unique_lookup(
    dim_match,
    ["dim_state_norm", "dim_name_loose"],
    "airport_code"
)

# New safer helper:
# If a city/state combination has exactly one airport in the dimension,
# then a DOT airport with that same city/state can safely inherit that airport code.
lookup_state_city_unique = build_unique_lookup(
    dim_match,
    ["dim_state_norm", "dim_city_norm"],
    "airport_code"
)

lookup_name_global = build_unique_lookup(
    dim_match,
    ["dim_name_norm"],
    "airport_code"
)

lookup_name_loose_global = build_unique_lookup(
    dim_match,
    ["dim_name_loose"],
    "airport_code"
)

dot_to_iata = {}
match_method = {}

for _, row in dot_candidates.iterrows():
    dot_id = row["DOT_ID"]

    keys_to_try = [
        (
            "state_city_name_exact",
            lookup_state_city_name,
            (row["dot_state_norm"], row["dot_city_norm"], row["dot_name_norm"])
        ),
        (
            "state_name_exact",
            lookup_state_name,
            (row["dot_state_norm"], row["dot_name_norm"])
        ),
        (
            "state_name_loose",
            lookup_state_name_loose,
            (row["dot_state_norm"], row["dot_name_loose"])
        ),
        (
            "state_city_unique",
            lookup_state_city_unique,
            (row["dot_state_norm"], row["dot_city_norm"])
        ),
        (
            "global_name_exact",
            lookup_name_global,
            (row["dot_name_norm"],)
        ),
        (
            "global_name_loose",
            lookup_name_loose_global,
            (row["dot_name_loose"],)
        ),
    ]

    for method, lookup, key in keys_to_try:
        if key in lookup:
            dot_to_iata[dot_id] = lookup[key]
            match_method[dot_id] = method
            break


# Fuzzy match remaining DOT names within the same state only
unmatched_after_exact = sorted(set(numeric_fact_codes_in_dot_lookup) - set(dot_to_iata))

for dot_id in unmatched_after_exact:
    dot_row = dot_candidates.loc[dot_candidates["DOT_ID"] == dot_id].iloc[0]

    target_name = dot_row["dot_name_loose"]
    target_state = dot_row["dot_state_norm"]

    if pd.isna(target_name) or pd.isna(target_state):
        continue

    state_candidates = dim_match[
        (dim_match["dim_state_norm"] == target_state)
        & dim_match["dim_name_loose"].notna()
    ].copy()

    if state_candidates.empty:
        continue

    scored = []
    for _, dim_row in state_candidates.iterrows():
        score = SequenceMatcher(
            None,
            str(target_name),
            str(dim_row["dim_name_loose"])
        ).ratio()

        scored.append((score, dim_row["airport_code"]))

    scored = sorted(scored, reverse=True)

    if not scored:
        continue

    best_score, best_code = scored[0]
    second_score = scored[1][0] if len(scored) > 1 else 0

    # High threshold + margin reduces risky false matches
    if best_score >= 0.92 and (best_score - second_score) >= 0.03:
        dot_to_iata[dot_id] = best_code
        match_method[dot_id] = "state_fuzzy_name"


# ------------------------------------------------------------
# 6B. Manual verified DOT -> IATA overrides
# ------------------------------------------------------------
# These are known airport aliases/renames where the DOT lookup name and
# dim_airports name refer to the same airport but wording differs too much
# for conservative automatic matching.
#
# Only keep overrides when:
# - the DOT airport name and IATA airport name are clearly the same airport
# - the IATA code exists in dim_airports
# - the DOT ID appears in the current fact data / DOT lookup

manual_dot_to_iata_overrides_all = {
    "10157": "ACV",  # California Redwood Coast Humboldt County -> Arcata/Eureka ACV
    "10208": "AGS",  # Augusta Regional at Bush Field -> Augusta Regional Airport (Bush Field)
    "10372": "ASE",  # Aspen Pitkin County Sardy Field -> Aspen-Pitkin County Airport
    "10577": "BGM",  # Greater Binghamton/Edwin A. Link Field -> Greater Binghamton Airport
    "10685": "BMI",  # Central IL Regional Airport at Bloomington -> Central Illinois Regional
    "10713": "BOI",  # Boise Air Terminal -> Boise Airport (Boise Air Terminal)
    "10721": "BOS",  # Logan International -> Gen. Edward Lawrence Logan International
    "10781": "BTR",  # Baton Rouge Metropolitan/Ryan Field -> Baton Rouge Metropolitan
    "10785": "BTV",  # Patrick Leahy Burlington International -> Burlington International
    "10821": "BWI",  # Baltimore/Washington International Thurgood Marshall -> BWI
    "10926": "CDV",  # Merle K Mudhole Smith -> Merle K. Smith Airport
    "10980": "CHA",  # Lovell Field -> Chattanooga Airport
    "10994": "CHS",  # Charleston AFB/International -> Charleston International / AFB
    "11066": "CMH",  # John Glenn Columbus International -> Port Columbus / CMH
    "11122": "CPR",  # Casper/Natrona County International -> Natrona County International
    "11146": "CRW",  # West Virginia International Yeager -> Yeager Airport
    "11433": "DTW",  # Detroit Metro Wayne County -> Detroit Metropolitan
    "11577": "ERI",  # Erie International/Tom Ridge Field -> Erie International
    "11603": "EUG",  # Mahlon Sweet Field -> Eugene Airport
    "11641": "FAY",  # Fayetteville Regional/Grannis Field -> Fayetteville Regional
    "11775": "FSD",  # Joe Foss Field -> Sioux Falls Regional
    "11865": "GCC",  # Northeast Wyoming Regional -> Gillette-Campbell County
    "11982": "GRK",  # Robert Gray AAF -> Killeen-Fort Hood / GRK
    "12173": "HNL",  # Daniel K Inouye International -> Honolulu International
    "12217": "HSV",  # Huntsville International-Carl T Jones Field -> HSV
    "12266": "IAH",  # George Bush Intercontinental/Houston -> IAH
    "12343": "INL",  # Falls International Einarson Field -> Falls International
    "12448": "JAN",  # Jackson Medgar Wiley Evers International -> Jackson-Evers
    "12758": "KOA",  # Ellison Onizuka Kona International at Keahole -> KOA
    "12889": "LAS",  # Harry Reid International -> Las Vegas airport code LAS
    "12951": "LFT",  # Lafayette Regional Paul Fournet Field -> Lafayette Regional
    "12992": "LIT",  # Bill and Hillary Clinton Nat Adams Field -> Clinton National
    "13158": "MAF",  # Midland International Air and Space Port -> Midland International
    "13241": "MEI",  # Key Field -> Meridian / MEI
    "13256": "MFE",  # McAllen International -> McAllen-Miller / MFE
    "13264": "MFR",  # Rogue Valley International - Medford -> Rogue Valley International
    "13360": "MLB",  # Melbourne Orlando International -> Melbourne International
    "13367": "MLI",  # Quad Cities International -> Quad City International
    "13459": "MQT",  # Marquette Sawyer Regional -> Sawyer International
    "13485": "MSN",  # Dane County Regional-Truax Field -> Dane County Regional
    "13486": "MSO",  # Missoula Montana -> Missoula International
    "13851": "OKC",  # OKC Will Rogers International -> Will Rogers World
    "14108": "PIA",  # General Downing - Peoria International -> Peoria International
    "14150": "PLN",  # Pellston Regional Emmet County -> Pellston Regional Airport of Emmet County
    "14307": "PVD",  # Rhode Island TF Green International -> Theodore Francis Green
    "14489": "RDM",  # Roberts Field -> Redmond Municipal Airport (Roberts Field)
    "14543": "RKS",  # Southwest Wyoming Regional -> Rock Springs-Sweetwater County
    "14574": "ROA",  # Roanoke Blacksburg Regional -> Roanoke Regional
    "14576": "ROC",  # Frederick Douglass Greater Rochester International -> Greater Rochester
    "14711": "SCE",  # State College Regional -> University Park / State College
    "14730": "SDF",  # Louisville Muhammad Ali International -> Louisville International
    "14842": "SJT",  # San Angelo Regional/Mathis Field -> San Angelo Regional
    "14905": "SMX",  # Santa Maria Public/Capt. G. Allan Hancock Field -> SMX
    "14908": "SNA",  # John Wayne Airport-Orange County -> John Wayne Airport
    "14960": "SPS",  # Sheppard AFB/Wichita Falls Municipal -> SPS
    "15048": "SUX",  # Sioux Gateway Brig Gen Bud Day Field -> Sioux Gateway
    "15070": "SWF",  # New York Stewart International -> Stewart International
    "15295": "TOL",  # Eugene F Kranz Toledo Express -> Toledo Express
    "15389": "TWF",  # Joslin Field - Magic Valley Regional -> Magic Valley Regional
    "15401": "TXK",  # Texarkana Regional-Webb Field -> Texarkana Regional
    "15624": "VPS",  # Eglin AFB Destin Fort Walton Beach -> VPS
    "15919": "XNA",  # Northwest Arkansas National -> Northwest Arkansas Regional
    "16218": "YUM",  # Yuma MCAS/Yuma International -> Yuma International
}

valid_iata_codes = set(dim_airports["airport_code"].astype("string").str.strip())

# Only apply overrides relevant to this dataset and existing DOT lookup
manual_dot_to_iata_overrides = {
    dot_id: iata_code
    for dot_id, iata_code in manual_dot_to_iata_overrides_all.items()
    if dot_id in set(numeric_fact_codes_in_dot_lookup)
}

invalid_overrides = {
    dot_id: iata_code
    for dot_id, iata_code in manual_dot_to_iata_overrides.items()
    if iata_code not in valid_iata_codes
}

if invalid_overrides:
    raise ValueError(f"Manual override points to unknown airport_code in dim_airports: {invalid_overrides}")

manual_conflicts = {
    dot_id: {
        "automatic": dot_to_iata[dot_id],
        "manual": iata_code
    }
    for dot_id, iata_code in manual_dot_to_iata_overrides.items()
    if dot_id in dot_to_iata and dot_to_iata[dot_id] != iata_code
}

if manual_conflicts:
    print("\nManual overrides replacing automatic matches:")
    print(manual_conflicts)

for dot_id, iata_code in manual_dot_to_iata_overrides.items():
    dot_to_iata[dot_id] = iata_code
    match_method[dot_id] = "manual_verified_alias"


dot_to_iata_df = pd.DataFrame({
    "DOT_ID": list(dot_to_iata.keys()),
    "IATA_CODE": list(dot_to_iata.values()),
    "match_method": [match_method[k] for k in dot_to_iata.keys()]
})

print("\nDOT IDs mapped to existing IATA airport codes after all matching:", len(dot_to_iata_df))
print(dot_to_iata_df["match_method"].value_counts(dropna=False))


# ------------------------------------------------------------
# 7. Apply canonical airport keys to fact table
#    - Existing IATA stays as-is
#    - Numeric DOT mapped to IATA where safe
#    - Remaining numeric DOT stays as DOT ID but gets named in dimension
# ------------------------------------------------------------

def resolve_airport_key(series):
    raw = series.astype("string").str.strip()

    mapped = raw.map(dot_to_iata)

    resolved = raw.copy()
    resolved = resolved.where(mapped.isna(), mapped)

    return resolved.astype("string")


fact_flights["ORIGIN_AIRPORT"] = resolve_airport_key(fact_flights["ORIGIN_AIRPORT_RAW"])
fact_flights["DESTINATION_AIRPORT"] = resolve_airport_key(fact_flights["DESTINATION_AIRPORT_RAW"])

fact_flights["origin_airport_mapping_status"] = "KNOWN_OR_UNCHANGED"
fact_flights.loc[
    fact_flights["ORIGIN_AIRPORT_RAW"].isin(dot_to_iata.keys()),
    "origin_airport_mapping_status"
] = "DOT_TO_IATA"

fact_flights.loc[
    fact_flights["ORIGIN_AIRPORT_RAW"].str.fullmatch(r"\d+").fillna(False)
    & ~fact_flights["ORIGIN_AIRPORT_RAW"].isin(dot_to_iata.keys()),
    "origin_airport_mapping_status"
] = "DOT_ID_RETAINED"

fact_flights["destination_airport_mapping_status"] = "KNOWN_OR_UNCHANGED"
fact_flights.loc[
    fact_flights["DESTINATION_AIRPORT_RAW"].isin(dot_to_iata.keys()),
    "destination_airport_mapping_status"
] = "DOT_TO_IATA"

fact_flights.loc[
    fact_flights["DESTINATION_AIRPORT_RAW"].str.fullmatch(r"\d+").fillna(False)
    & ~fact_flights["DESTINATION_AIRPORT_RAW"].isin(dot_to_iata.keys()),
    "destination_airport_mapping_status"
] = "DOT_ID_RETAINED"


# ------------------------------------------------------------
# 8. Build final airport dimension with NO blank airport names
# ------------------------------------------------------------

dim_airports_fixed = dim_airports.copy()

if "mapping_status" not in dim_airports_fixed.columns:
    dim_airports_fixed["mapping_status"] = "KNOWN_AIRPORT"

# Airport keys now used by fact table after DOT -> IATA mapping
fact_airport_codes_final = pd.concat([
    fact_flights["ORIGIN_AIRPORT"],
    fact_flights["DESTINATION_AIRPORT"]
]).dropna().astype("string").str.strip().unique()

dim_airport_codes_base = set(dim_airports_fixed["airport_code"].dropna().astype("string").str.strip())

still_missing_codes = sorted(set(fact_airport_codes_final) - dim_airport_codes_base)

# For numeric DOT IDs still remaining, use DOT lookup for readable names
still_missing_dot = [code for code in still_missing_codes if bool(re.fullmatch(r"\d+", str(code)))]
still_missing_other = [code for code in still_missing_codes if code not in still_missing_dot]

dot_rows = airport_lookup_DOT[
    airport_lookup_DOT["DOT_ID"].isin(still_missing_dot)
].copy()

dot_dim_rows = pd.DataFrame({
    "airport_code": dot_rows["DOT_ID"].astype("string"),
    "airport_name": dot_rows["dot_airport_name"].fillna(dot_rows["AIRPORT_INFO"]),
    "city": dot_rows["dot_city"],
    "state": dot_rows["dot_state"],
    "country": "USA",
    "latitude": pd.NA,
    "longitude": pd.NA,
    "mapping_status": "DOT_LOOKUP_ONLY"
})

# Fallback only if truly not found in DOT lookup
dot_resolved_codes = set(dot_dim_rows["airport_code"].dropna().astype("string"))
unresolved_codes = sorted(set(still_missing_codes) - dot_resolved_codes)

fallback_rows = pd.DataFrame({
    "airport_code": unresolved_codes,
    "airport_name": [f"Unresolved Airport ({code})" for code in unresolved_codes],
    "city": pd.NA,
    "state": pd.NA,
    "country": pd.NA,
    "latitude": pd.NA,
    "longitude": pd.NA,
    "mapping_status": "UNRESOLVED"
})

dim_airports_fixed = pd.concat(
    [dim_airports_fixed, dot_dim_rows, fallback_rows],
    ignore_index=True
)

dim_airports_fixed = (
    dim_airports_fixed
    .drop_duplicates(subset=["airport_code"], keep="first")
    .reset_index(drop=True)
)

# Final safety: no blank display names
dim_airports_fixed["airport_name"] = dim_airports_fixed["airport_name"].fillna(
    "Unresolved Airport (" + dim_airports_fixed["airport_code"].astype("string") + ")"
)


# ------------------------------------------------------------
# 9. Final validation
# ------------------------------------------------------------

final_dim_codes = set(dim_airports_fixed["airport_code"].dropna().astype("string").str.strip())

final_fact_codes = set(
    pd.concat([
        fact_flights["ORIGIN_AIRPORT"],
        fact_flights["DESTINATION_AIRPORT"]
    ])
    .dropna()
    .astype("string")
    .str.strip()
)

missing_after_fix = sorted(final_fact_codes - final_dim_codes)

print("\nFinal validation")
print("Final dim_airports_fixed shape:", dim_airports_fixed.shape)
print("Airport keys in fact but missing from dim:", len(missing_after_fix))
print("Blank airport names in dim_airports_fixed:", int(dim_airports_fixed["airport_name"].isna().sum()))

print("\nFact mapping status - origin:")
print(fact_flights["origin_airport_mapping_status"].value_counts(dropna=False))

print("\nFact mapping status - destination:")
print(fact_flights["destination_airport_mapping_status"].value_counts(dropna=False))

print("\nAirport dimension mapping status:")
print(dim_airports_fixed["mapping_status"].value_counts(dropna=False))

print("\nRemaining numeric origin rows after mapping:")
print(int(fact_flights["ORIGIN_AIRPORT"].astype("string").str.fullmatch(r"\d+").fillna(False).sum()))

print("\nRemaining numeric destination rows after mapping:")
print(int(fact_flights["DESTINATION_AIRPORT"].astype("string").str.fullmatch(r"\d+").fillna(False).sum()))

print("\nDOT -> IATA match methods:")
display(
    dot_to_iata_df
    .sort_values(["match_method", "DOT_ID"])
    .reset_index(drop=True)
)

print("\nSample DOT lookup-only airports:")
display(dim_airports_fixed[dim_airports_fixed["mapping_status"] == "DOT_LOOKUP_ONLY"].head(20))

if missing_after_fix:
    print("\nWARNING: Some fact airport keys still missing from dim:")
    print(missing_after_fix[:30])
else:
    print("\nSUCCESS: every airport key used in fact_flights exists in dim_airports_fixed.")

Airport code repair: DOT IDs, aliases, and final dimension coverage
Unique numeric airport codes in fact: 307
Numeric codes found in DOT lookup: 307
Numeric codes NOT found in DOT lookup: 0

DOT IDs mapped to existing IATA airport codes after all matching: 307
match_method
state_name_loose         226
manual_verified_alias     63
state_city_name_exact     12
global_name_loose          5
state_name_exact           1
Name: count, dtype: int64

Final validation
Final dim_airports_fixed shape: (322, 8)
Airport keys in fact but missing from dim: 0
Blank airport names in dim_airports_fixed: 0

Fact mapping status - origin:
origin_airport_mapping_status
KNOWN_OR_UNCHANGED    5332914
DOT_TO_IATA            486165
Name: count, dtype: int64

Fact mapping status - destination:
destination_airport_mapping_status
KNOWN_OR_UNCHANGED    5332914
DOT_TO_IATA            486165
Name: count, dtype: int64

Airport dimension mapping status:
mapping_status
KNOWN_AIRPORT    322
Name: count, dtype: int64

Rema

,DOT_ID,IATA_CODE,match_method
0,11193,CVG,global_name_loose
1,11278,DCA,global_name_loose
2,12016,GUM,global_name_loose
3,12264,IAD,global_name_loose
4,14222,PPG,global_name_loose
...,...,...,...
302,15380,TVC,state_name_loose
303,15411,TYR,state_name_loose
304,15412,TYS,state_name_loose
305,15497,UST,state_name_loose



Sample DOT lookup-only airports:


,airport_code,airport_name,city,state,country,latitude,longitude,mapping_status



SUCCESS: every airport key used in fact_flights exists in dim_airports_fixed.


## 14 — Export Star Schema Outputs

In [29]:
save_parquet(fact_flights, PROCESSED_DIR / "fact_flights.parquet")
save_csv(dim_airlines, PROCESSED_DIR / "dim_airlines.csv")
#save_csv(dim_airports, PROCESSED_DIR / "dim_airports.csv")
save_csv(dim_airports_fixed, PROCESSED_DIR / "dim_airports.csv")
save_csv(dim_cancel, PROCESSED_DIR / "dim_cancellation_codes.csv")
save_csv(dim_delay_driver, PROCESSED_DIR / "dim_delay_driver.csv")
print("\nStep 2 complete. Star schema outputs saved.")

Saved Parquet: C:\Users\kados\OneDrive\Desktop\folders\Data Analysis Portfolio\Airline Flight Delays Portfolio\data\processed\fact_flights.parquet | rows=5,819,079 cols=40
Saved CSV: C:\Users\kados\OneDrive\Desktop\folders\Data Analysis Portfolio\Airline Flight Delays Portfolio\data\processed\dim_airlines.csv | rows=14 cols=2
Saved CSV: C:\Users\kados\OneDrive\Desktop\folders\Data Analysis Portfolio\Airline Flight Delays Portfolio\data\processed\dim_airports.csv | rows=322 cols=8
Saved CSV: C:\Users\kados\OneDrive\Desktop\folders\Data Analysis Portfolio\Airline Flight Delays Portfolio\data\processed\dim_cancellation_codes.csv | rows=4 cols=2
Saved CSV: C:\Users\kados\OneDrive\Desktop\folders\Data Analysis Portfolio\Airline Flight Delays Portfolio\data\processed\dim_delay_driver.csv | rows=6 cols=3

Step 2 complete. Star schema outputs saved.


## Step 2 Summary

This notebook builds the curated analytical model for the Power BI dashboard.

**Generated outputs:**
- `data/processed/fact_flights.parquet` — main fact table with KPI flags
- `data/processed/dim_airlines.csv` — airline lookup dimension
- `data/processed/dim_airports.csv` — airport lookup dimension
- `data/processed/dim_cancellation_codes.csv` — cancellation reason lookup dimension

**KPI flags derived:**
- `is_otp15_eligible` — completed flights (not cancelled, not diverted)
- `is_on_time_otp15` — On-Time Performance (arrival delay ≤ 15 mins)
- `is_delayed_otp15` — delayed (arrival delay > 15 mins)
- `is_severe_delay_sd60` — Severe Delay (arrival delay ≥ 60 mins)
- `primary_delay_driver` — attribution of delay cause
- `operational_disruption_flag` — cancelled, diverted, or severely delayed